# 🔐 Gestión de permisos de carpetas del workspace

Aplica ACLs granulares sobre carpetas de `/Shared` usando la **Permissions API** de
Databricks, con validación previa, modo simulación y verificación de cambios.

---

## Requisitos

- Ejecutar con un usuario **Workspace Admin** (o con `CAN_MANAGE` sobre las carpetas destino).
- No hace falta configurar host ni token: se derivan del contexto del notebook.

## Niveles de permiso

| Nivel | Ver | Ejecutar | Editar | Crear/borrar/mover | Cambiar permisos |
|---|:-:|:-:|:-:|:-:|:-:|
| `CAN_READ` | ✓ | | | | |
| `CAN_RUN` | ✓ | ✓ | | | |
| `CAN_EDIT` | ✓ | ✓ | ✓ | | |
| `CAN_MANAGE` | ✓ | ✓ | ✓ | ✓ | ✓ |

## Flujo

| # | Sección | Qué hace | Escribe |
|---|---|---|:-:|
| 1 | Setup | Deriva host/token, crea el widget de modo | |
| 2 | Configuración | Define carpetas y permisos | |
| 3 | Diagnóstico | Lista el workspace para hallar rutas reales | |
| 4 | Resolver | Traduce rutas a `object_id` + `object_type` | |
| 5 | Estado actual | Muestra las ACLs vigentes | |
| 6 | Validación | Comprueba en SCIM que los principals existan | |
| 7 | Aplicar | Ejecuta el cambio según el widget | ⚠️ sí |
| 8 | Verificación | Diff antes/después | |

---

## ⚠️ Antes de usarlo, tres advertencias

**1. `PUT` reemplaza la ACL completa, no la suma.**
Cualquier grupo o usuario que hoy tenga acceso y no aparezca en `FOLDERS_CONFIG`
**pierde el acceso**. Revisa siempre la sección 5 y el dry-run antes de aplicar.

**2. Las carpetas de bundles se sobrescriben en cada deploy.**
Las carpetas con `databricks.yml`, `state/` y `artifacts/` son despliegues de
Databricks Asset Bundles. El pipeline de Azure DevOps puede pisar estos permisos
en el próximo deploy. Para gobierno permanente usa el bloque `permissions:` del
`databricks.yml` de cada bundle; este notebook sirve para carpetas manuales o
correcciones puntuales.

**3. `/Shared` raíz no admite cambios de ACL.**
La API responde `400 Cannot modify permissions of directory`. Es una restricción de
diseño de Databricks, no un problema de privilegios. Aplica sobre las subcarpetas.


## 1. Setup

Deriva `host` y `token` del contexto de ejecución — esto evita el error
`Invalid access to Org` que aparece al hardcodear el host de otro workspace.

También crea el **widget `modo`** en la barra superior del notebook, que controla si la
sección 7 simula o aplica de verdad. Al ser un `dropdown`, conserva el valor elegido
entre ejecuciones de esta celda.


In [ ]:
import json
import requests

# ── Host y token derivados del contexto del notebook ──
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()
token = ctx.apiToken().get()
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# ── Widget de modo de ejecución ──
dbutils.widgets.dropdown(
    name="modo",
    defaultValue="DRY-RUN",
    choices=["DRY-RUN", "APLICAR"],
    label="Modo de ejecucion",
)

# ── Sanity check ──
_r = requests.get(
    f"{host}/api/2.0/workspace/get-status",
    headers=headers,
    params={"path": "/Shared"},
)
_j = _r.json()

print(f"Host  : {host}")
print(f"Estado: /Shared -> {_r.status_code} {_j.get('object_type', _j.get('message'))}")
print(f"Modo  : {dbutils.widgets.get('modo')}   (cambialo en el widget de arriba)")


## 2. Configuración de carpetas y permisos

Cada entrada es `"ruta": {NIVEL: [entidades]}`. Las entidades se identifican con
**una** de estas claves:

| Clave | Valor esperado |
|---|---|
| `group_name` | nombre del grupo tal cual aparece en SCIM |
| `user_name` | email del usuario |
| `service_principal_name` | ⚠️ el **applicationId (UUID)**, no el display name |

Las rutas se verificaron contra el workspace con la sección 3. Si cambian, vuelve a
correr el diagnóstico: la API responde `Path doesn't exist` ante cualquier desajuste.


In [ ]:
FOLDERS_CONFIG = {
    # ─── Bundle de Arquitectura de Datos (dev) ───
    "/Shared/databricks_repo_ad_dev": {
        "CAN_MANAGE": [
            {"group_name": "admins"},
            {"group_name": "GS_ADMINISTRADORCATALOGO_CORONA"},
            {"group_name": "GS_ARQUITECTODATOS_CORONA"},
            {"service_principal_name": "sp-databricks-prd-contributor-arqanalitica"},
        ],
        "CAN_EDIT": [
            {"group_name": "GS_ANALISTADATOS_CORONA"},
            {"group_name": "GS_CIENTIFICODATOS_CORONA"},
        ],
        "CAN_READ": [
            {"group_name": "users"},
        ],
    },

    # ─── Bundle de Analitica / Ingenieria (dev) ───
    "/Shared/databricks_repo_aq_dev": {
        "CAN_MANAGE": [
            {"group_name": "admins"},
            {"group_name": "GS_ADMINISTRADORCATALOGO_CORONA"},
            {"group_name": "GS_ARQUITECTODATOS_CORONA"},
            {"service_principal_name": "sp-databricks-prd-contributor-arqanalitica"},
        ],
        "CAN_EDIT": [
            {"group_name": "GS_INGENIERODATOS_CORONA"},
            {"group_name": "GS_INGENIERODATOS_IDATA"},
        ],
        "CAN_READ": [
            {"group_name": "users"},
        ],
    },

    # ─── Pendientes: descomenta cuando definas los grupos ───
    # "/Shared/databricks_repo_gb_dev": {...},   # id 772919698670255
    # "/Shared/databricks_repo_dev":    {...},   # id 772919698669634
}

NIVELES_VALIDOS = {"CAN_READ", "CAN_RUN", "CAN_EDIT", "CAN_MANAGE"}
CLAVES_VALIDAS = {"group_name", "user_name", "service_principal_name"}


def validar_estructura(config):
    """Revisa la forma de FOLDERS_CONFIG antes de tocar la API."""
    errores = []
    for path, niveles in config.items():
        if not path.startswith("/"):
            errores.append(f"{path}: la ruta debe empezar con /")
        for nivel, entidades in niveles.items():
            if nivel not in NIVELES_VALIDOS:
                errores.append(f"{path}: nivel desconocido '{nivel}'")
            for e in entidades:
                claves = set(e) & CLAVES_VALIDAS
                if len(claves) != 1:
                    errores.append(f"{path} / {nivel}: entidad invalida {e}")
    return errores


_errores = validar_estructura(FOLDERS_CONFIG)
if _errores:
    for e in _errores:
        print(f"  ERROR  {e}")
    raise ValueError(f"{len(_errores)} error(es) de estructura en FOLDERS_CONFIG")

_total = sum(len(v) for niveles in FOLDERS_CONFIG.values() for v in niveles.values())
print(f"Estructura valida: {len(FOLDERS_CONFIG)} carpeta(s), {_total} asignacion(es)")
for path in FOLDERS_CONFIG:
    print(f"   - {path}")


## 3. Diagnóstico — explorar el workspace

Lista lo que existe realmente, con `path`, `object_type` y `object_id`. Úsala cuando
una ruta falle con `Path doesn't exist` o cuando no sepas el nombre exacto de una carpeta.

Los objetos **REPO** se muestran para que tengas el panorama completo, pero este notebook
no los gestiona: se administran por fuera.

*Solo lectura.*


In [ ]:
ICONOS = {
    "DIRECTORY": "[DIR ]",
    "REPO": "[REPO]",
    "NOTEBOOK": "[NB  ]",
    "FILE": "[FILE]",
}


def listar(path, nivel=0, max_nivel=1):
    """Lista recursivamente el contenido de una ruta del workspace."""
    resp = requests.get(
        f"{host}/api/2.0/workspace/list",
        headers=headers,
        params={"path": path},
    )
    if resp.status_code != 200:
        print(f"{'  ' * nivel}ERROR {path}: {resp.json().get('message', resp.text)}")
        return

    objetos = sorted(resp.json().get("objects", []), key=lambda o: o["path"])
    if not objetos:
        print(f"{'  ' * nivel}(vacio)")
        return

    for obj in objetos:
        tipo = obj["object_type"]
        icono = ICONOS.get(tipo, "[????]")
        print(f"{'  ' * nivel}{icono} {obj['path']:<55} id={obj.get('object_id', '-')}")
        if tipo == "DIRECTORY" and nivel < max_nivel:
            listar(obj["path"], nivel + 1, max_nivel)


for raiz in ["/Shared", "/Repos"]:
    print("=" * 90)
    print(f" CONTENIDO DE {raiz}")
    print("=" * 90)
    listar(raiz)
    print()

print("Copia los paths exactos que necesites a FOLDERS_CONFIG (seccion 2).")
print("Solo se gestionan objetos DIRECTORY; los REPO se omiten.")


## 4. Resolver objetos

Traduce cada ruta a su `object_id` y `object_type`, porque la Permissions API trabaja
con IDs y **el endpoint depende del tipo**:

```
DIRECTORY  ->  /api/2.0/permissions/directories/{id}
NOTEBOOK   ->  /api/2.0/permissions/notebooks/{id}
```

Lo que no sea uno de esos tipos se omite con un aviso, sin cortar la ejecución.


In [ ]:
# Los objetos REPO se ignoran a proposito: se gestionan por fuera de este notebook.
ENDPOINT_POR_TIPO = {
    "DIRECTORY": "directories",
    "NOTEBOOK": "notebooks",
}


def resolve_object(path):
    """Devuelve (object_id, object_type) de un path del workspace, o (None, None)."""
    resp = requests.get(
        f"{host}/api/2.0/workspace/get-status",
        headers=headers,
        params={"path": path},
    )
    if resp.status_code != 200:
        print(f"  ERROR  {path}: {resp.json().get('message', resp.text)}")
        return None, None

    data = resp.json()
    tipo = data.get("object_type")
    if tipo not in ENDPOINT_POR_TIPO:
        print(f"  OMITIDO {path}: es {tipo}")
        return None, None

    return str(data["object_id"]), tipo


def permissions_url(object_id, object_type):
    """URL del endpoint de permisos segun el tipo de objeto."""
    return f"{host}/api/2.0/permissions/{ENDPOINT_POR_TIPO[object_type]}/{object_id}"


print("=" * 90)
print(" RESOLVIENDO OBJETOS DEL WORKSPACE")
print("=" * 90)

objetos = {}   # path -> (object_id, object_type)
for path in FOLDERS_CONFIG:
    oid, tipo = resolve_object(path)
    if oid:
        objetos[path] = (oid, tipo)
        print(f"  OK     {path:<45} {tipo:<10} id={oid}")

print(f"\n{len(objetos)}/{len(FOLDERS_CONFIG)} objetos resueltos")
if len(objetos) < len(FOLDERS_CONFIG):
    print("Corre la seccion 3 para encontrar las rutas correctas.")


## 5. Estado actual

Muestra las ACLs vigentes y guarda un **snapshot** en `permisos_antes`, que la
sección 8 usa para calcular el diff.

Presta atención a la columna *Herencia*: los permisos `heredado` vienen del padre y no
se pierden al reemplazar la ACL; los `directo` sí son los que este notebook sobrescribe.


In [ ]:
def leer_acl(object_id, object_type):
    """Devuelve {entidad: [niveles directos]} o None si falla la lectura."""
    resp = requests.get(permissions_url(object_id, object_type), headers=headers)
    if resp.status_code != 200:
        return None

    acl = {}
    for e in resp.json().get("access_control_list", []):
        nombre = (e.get("group_name")
                  or e.get("user_name")
                  or e.get("service_principal_name")
                  or "???")
        directos = sorted({
            p.get("permission_level")
            for p in (e.get("all_permissions") or [])
            if not p.get("inherited", False) and p.get("permission_level")
        })
        if directos:
            acl[nombre] = directos
    return acl


def mostrar_permisos(path, object_id, object_type):
    """Imprime la tabla de permisos actuales de un objeto."""
    resp = requests.get(permissions_url(object_id, object_type), headers=headers)

    print(f"\n{'-' * 90}")
    print(f" {path}  [{object_type}] id={object_id}")
    print(f"{'-' * 90}")

    if resp.status_code != 200:
        print(f"  ERROR leyendo permisos: {resp.status_code} {resp.text[:200]}")
        return

    print(f"  {'Grupo/Entidad':<45} {'Permiso':<15} Herencia")
    print(f"  {'-' * 78}")

    for e in resp.json().get("access_control_list", []):
        nombre = (e.get("group_name")
                  or e.get("user_name")
                  or e.get("service_principal_name")
                  or "???")
        permisos = e.get("all_permissions") or []
        if not permisos:
            print(f"  {nombre:<45} {'(sin permisos)':<15} -")
            continue
        for p in permisos:
            nivel = p.get("permission_level", "N/A")
            herencia = "heredado" if p.get("inherited", False) else "directo"
            print(f"  {nombre:<45} {nivel:<15} {herencia}")


print("=" * 90)
print(" PERMISOS ACTUALES (ANTES DEL CAMBIO)")
print("=" * 90)

permisos_antes = {}
for path, (oid, tipo) in objetos.items():
    mostrar_permisos(path, oid, tipo)
    permisos_antes[path] = leer_acl(oid, tipo)

print(f"\nSnapshot guardado para {len(permisos_antes)} objeto(s).")


## 6. Validación de principals

El dry-run solo imprime el payload; no comprueba que las entidades existan. Un nombre
mal escrito hace fallar el `PUT` **completo** de esa carpeta, porque la operación es atómica.

Esta celda consulta SCIM por cada grupo, usuario y service principal de la configuración.

> ⚠️ En `service_principal_name` la Permissions API espera el **`applicationId` (UUID)**,
> no el display name. Si un SP sale como `FALTA`, la celda lista los del workspace con su UUID.

*Solo lectura.*


In [ ]:
def _existe_en_scim(recurso, campo, valor):
    """True si SCIM devuelve al menos un resultado para: campo eq valor."""
    filtro = '{} eq "{}"'.format(campo, valor)
    resp = requests.get(
        f"{host}/api/2.0/preview/scim/v2/{recurso}",
        headers=headers,
        params={"filter": filtro},
    )
    if resp.status_code != 200:
        print(f"  ERROR consultando {recurso}: {resp.status_code} {resp.text[:200]}")
        return False
    return resp.json().get("totalResults", 0) > 0


SCIM_POR_CLAVE = {
    "group_name": ("Groups", "displayName", "group"),
    "service_principal_name": ("ServicePrincipals", "applicationId", "sp"),
    "user_name": ("Users", "userName", "user"),
}


def validar_principals(config):
    """Verifica que cada principal de la config exista. Devuelve los que faltan."""
    faltantes = []
    vistos = set()

    for niveles in config.values():
        for entidades in niveles.values():
            for e in entidades:
                clave = tuple(sorted(e.items()))
                if clave in vistos:
                    continue
                vistos.add(clave)

                for campo_config, (recurso, campo_scim, etiqueta) in SCIM_POR_CLAVE.items():
                    if campo_config in e:
                        valor = e[campo_config]
                        existe = _existe_en_scim(recurso, campo_scim, valor)
                        print(f"  {'OK   ' if existe else 'FALTA'}  {etiqueta:<6} {valor}")
                        if not existe:
                            faltantes.append(f"{etiqueta} {valor}")
                        break

    return faltantes


print("=" * 90)
print(" VALIDANDO GRUPOS Y SERVICE PRINCIPALS")
print("=" * 90)
faltantes = validar_principals(FOLDERS_CONFIG)

print()
if not faltantes:
    print("Todos los principals existen. Puedes continuar a la seccion 7.")
else:
    print(f"{len(faltantes)} principal(es) NO encontrados: el PUT fallaria.")
    print("Corrige FOLDERS_CONFIG antes de aplicar.\n")
    print("-" * 90)
    print(" SERVICE PRINCIPALS DEL WORKSPACE (usa el appId, no el displayName)")
    print("-" * 90)
    _r = requests.get(f"{host}/api/2.0/preview/scim/v2/ServicePrincipals", headers=headers)
    if _r.status_code == 200:
        for sp in _r.json().get("Resources", []):
            print(f"  {str(sp.get('displayName')):<50} appId={sp.get('applicationId')}")
    else:
        print(f"  ERROR: {_r.status_code} {_r.text[:200]}")


## 7. Aplicar permisos ⚠️

**Esta es la única sección que escribe.** El comportamiento lo decide el widget `modo`
de la barra superior:

| Widget | Efecto |
|---|---|
| `DRY-RUN` | Solo imprime el payload que se enviaría. No toca nada. |
| `APLICAR` | Ejecuta el `PUT` contra la Permissions API. |

Recuerda que `PUT` **reemplaza** la ACL: compara el dry-run con la sección 5 antes de
cambiar el widget. Si algo salió `FALTA` en la sección 6, corrígelo primero — la celda
se detiene sola si detecta principals inexistentes.


In [ ]:
def build_acl(config):
    """Construye el access_control_list a partir de la configuracion de una carpeta."""
    acl = []
    for permission_level, entidades in config.items():
        for entidad in entidades:
            entry = dict(entidad)          # copia: no mutar el original
            entry["permission_level"] = permission_level
            acl.append(entry)
    return acl


def nombre_entidad(entry):
    return (entry.get("group_name")
            or entry.get("user_name")
            or entry.get("service_principal_name")
            or "???")


def apply_permissions(path, object_id, object_type, config, dry_run=True):
    """Aplica permisos a un objeto. Con dry_run=True solo muestra lo que haria."""
    acl = build_acl(config)

    print(f"\n{'-' * 90}")
    print(f" {path}  [{object_type}] id={object_id}")
    print(f"{'-' * 90}")

    if dry_run:
        for entry in acl:
            print(f"     {nombre_entidad(entry):<45} -> {entry['permission_level']}")
        return True

    resp = requests.put(
        permissions_url(object_id, object_type),
        headers=headers,
        json={"access_control_list": acl},
    )
    if resp.status_code == 200:
        print("     Permisos aplicados exitosamente")
        return True

    print(f"     ERROR ({resp.status_code}): {resp.json().get('message', resp.text)}")
    return False


# ── El widget manda ──
MODO = dbutils.widgets.get("modo")
DRY_RUN = MODO != "APLICAR"

print("=" * 90)
print(f" MODO: {MODO}" + ("   (simulacion, no se escribe nada)" if DRY_RUN
                          else "   *** SE APLICARAN LOS CAMBIOS ***"))
print("=" * 90)

if not DRY_RUN and faltantes:
    raise RuntimeError(
        f"Hay {len(faltantes)} principal(es) inexistentes segun la seccion 6: {faltantes}. "
        "Corrige FOLDERS_CONFIG antes de aplicar."
    )

ok = 0
for path, (oid, tipo) in objetos.items():
    if apply_permissions(path, oid, tipo, FOLDERS_CONFIG[path], dry_run=DRY_RUN):
        ok += 1

print(f"\n{'=' * 90}")
if DRY_RUN:
    print(f"Dry-run completo: {ok}/{len(objetos)} objetos simulados")
    print("Para aplicar de verdad, pon el widget 'modo' en APLICAR y re-ejecuta esta celda.")
else:
    print(f"{ok}/{len(objetos)} objetos actualizados")
    print("Corre la seccion 8 para ver el diff.")


## 8. Verificación — diff antes/después

Compara el snapshot de la sección 5 contra el estado actual y reporta solo los
**permisos directos** que cambiaron:

- `+` permiso añadido
- `-` permiso retirado (⚠️ revisa que sea intencional)
- `~` nivel modificado

Si corriste la sección 7 en `DRY-RUN`, aquí no debería aparecer ningún cambio.


In [ ]:
def diff_acl(antes, despues):
    """Compara dos snapshots {entidad: [niveles]} y devuelve lineas de diff."""
    if antes is None or despues is None:
        return ["  (no se pudo comparar: fallo la lectura de permisos)"]

    lineas = []
    for entidad in sorted(set(antes) | set(despues)):
        a = antes.get(entidad)
        d = despues.get(entidad)
        if a == d:
            continue
        if a is None:
            lineas.append(f"  +  {entidad:<45} {', '.join(d)}")
        elif d is None:
            lineas.append(f"  -  {entidad:<45} {', '.join(a)}  (retirado)")
        else:
            lineas.append(f"  ~  {entidad:<45} {', '.join(a)} -> {', '.join(d)}")
    return lineas


print("=" * 90)
print(" VERIFICACION: DIFF DE PERMISOS DIRECTOS")
print("=" * 90)

sin_cambios = 0
for path, (oid, tipo) in objetos.items():
    despues = leer_acl(oid, tipo)
    lineas = diff_acl(permisos_antes.get(path), despues)

    print(f"\n{'-' * 90}")
    print(f" {path}")
    print(f"{'-' * 90}")
    if lineas:
        print("\n".join(lineas))
    else:
        print("  (sin cambios)")
        sin_cambios += 1

print(f"\n{'=' * 90}")
print(f"{len(objetos) - sin_cambios} objeto(s) con cambios, {sin_cambios} sin cambios")
print("\nEstado final detallado:")
for path, (oid, tipo) in objetos.items():
    mostrar_permisos(path, oid, tipo)
